# Market Depth

Adapted from [ib_async market_depth](https://ib-api-reloaded.github.io/ib_async/notebooks.html).
Demonstrates L2 order book data: depth exchanges, depth subscription, and live book display.

**Best run during market hours.**

In [ ]:
import os, threading, time
from dotenv import load_dotenv
from ibx import EWrapper, EClient, Contract

load_dotenv()
USERNAME = os.environ["IB_USERNAME"]
PASSWORD = os.environ["IB_PASSWORD"]

In [ ]:
class App(EWrapper):
    def __init__(self):
        super().__init__()
        self.connected = threading.Event()
        self.depth_exchanges = []
        self.depth_exchanges_done = threading.Event()
        self.bids = {}   # position -> (price, size, market_maker)
        self.asks = {}   # position -> (price, size, market_maker)

    def next_valid_id(self, order_id):
        self.connected.set()

    def mkt_depth_exchanges(self, depth_mkt_data_descriptions):
        self.depth_exchanges = list(depth_mkt_data_descriptions)
        self.depth_exchanges_done.set()

    def update_mkt_depth(self, req_id, position, operation, side, price, size):
        book = self.bids if side == 1 else self.asks
        if operation in (0, 1):  # insert or update
            book[position] = (price, size, "")
        elif operation == 2:     # delete
            book.pop(position, None)

    def update_mkt_depth_l2(self, req_id, position, market_maker, operation,
                            side, price, size, is_smart_depth):
        book = self.bids if side == 1 else self.asks
        if operation in (0, 1):
            book[position] = (price, size, market_maker)
        elif operation == 2:
            book.pop(position, None)

    def error(self, req_id, error_code, error_string, advanced_order_reject_json=""):
        if error_code not in (2104, 2106, 2158):
            print(f"Error {error_code}: {error_string}")

In [ ]:
app = App()
client = EClient(app)
client.connect(username=USERNAME, password=PASSWORD, paper=True)

thread = threading.Thread(target=client.run, daemon=True)
thread.start()
app.connected.wait(timeout=10)
print(f"Connected: {client.is_connected()}")

### Depth Exchanges

Query which exchanges support market depth:

In [ ]:
client.req_mkt_depth_exchanges()
app.depth_exchanges_done.wait(timeout=10)

print(f"{len(app.depth_exchanges)} exchanges support depth")
for desc in app.depth_exchanges[:10]:
    print(f"  {desc}")

### Subscribe to L2 Depth

Request 5 rows of AAPL order book on ISLAND (NASDAQ):

In [ ]:
aapl = Contract(con_id=265598, symbol="AAPL", sec_type="STK",
                exchange="ISLAND", currency="USD")

app.bids.clear()
app.asks.clear()
client.req_mkt_depth(1, aapl, 5, False, [])

time.sleep(3)
print(f"Book: {len(app.bids)} bid levels, {len(app.asks)} ask levels")

### Live Order Book (10 second loop)

Display the top-of-book with live updates:

In [ ]:
from IPython.display import clear_output

for _ in range(10):
    clear_output(wait=True)
    sorted_bids = sorted(app.bids.items())
    sorted_asks = sorted(app.asks.items())

    print("AAPL Order Book")
    print(f"{'Bid Size':>10} {'Bid':>10}  {'Ask':>10} {'Ask Size':>10}  {'MM':>6}")
    print("-" * 54)

    rows = max(len(sorted_bids), len(sorted_asks))
    for i in range(rows):
        if i < len(sorted_bids):
            _, (bp, bs, bmm) = sorted_bids[i]
            bid_str = f"{bs:>10.0f} {bp:>10.2f}"
        else:
            bid_str = f"{'':>10} {'':>10}"
        if i < len(sorted_asks):
            _, (ap, az, amm) = sorted_asks[i]
            ask_str = f"  {ap:>10.2f} {az:>10.0f}  {amm:>6}"
        else:
            ask_str = ""
        print(bid_str + ask_str)

    time.sleep(1)

### Cancel Depth Subscription

In [ ]:
client.cancel_mkt_depth(1)

### SmartDepth

Subscribe with `is_smart_depth=True` to get aggregated depth across exchanges.
Each `update_mkt_depth_l2` callback includes the originating exchange in `market_maker`:

In [ ]:
smart_aapl = Contract(con_id=265598, symbol="AAPL", sec_type="STK",
                      exchange="SMART", currency="USD")

app.bids.clear()
app.asks.clear()
client.req_mkt_depth(2, smart_aapl, 5, True, [])

time.sleep(3)

print(f"SmartDepth: {len(app.bids)} bid levels, {len(app.asks)} ask levels")
for pos in sorted(app.bids):
    price, size, mm = app.bids[pos]
    print(f"  Bid {pos}: {price:.2f} x {size:.0f}  [{mm}]")
for pos in sorted(app.asks):
    price, size, mm = app.asks[pos]
    print(f"  Ask {pos}: {price:.2f} x {size:.0f}  [{mm}]")

In [ ]:
client.cancel_mkt_depth(2)
client.disconnect()